In [1]:
# 可在 Jupyter Notebook 里直接运行

from __future__ import annotations
from pathlib import Path
import os
import fnmatch
from typing import Iterable, List, Tuple, Optional, Callable

def _should_ignore(name: str, ignore_globs: Optional[List[str]]) -> bool:
    if not ignore_globs:
        return False
    return any(fnmatch.fnmatch(name, pat) for pat in ignore_globs)

def _sort_key(p: Path) -> Tuple[int, str]:
    # 目录优先，再按名称不区分大小写排序
    return (0 if p.is_dir() else 1, p.name.lower())

def _children(
    p: Path,
    show_files: bool,
    follow_symlinks: bool,
    ignore_globs: Optional[List[str]],
) -> List[Path]:
    try:
        items = list(p.iterdir())
    except PermissionError:
        return []

    # 过滤隐藏/忽略项
    filtered = []
    for q in items:
        name = q.name
        if name.startswith(".") and not show_files:
            # 如果完全不想显示隐藏文件/夹，可以把 next 行替换为: continue
            pass
        if _should_ignore(name, ignore_globs):
            continue
        if q.is_dir() or show_files:
            filtered.append(q)

    # 如果不跟随符号链接目录，则把符号链接目录当作叶子
    if not follow_symlinks:
        out = []
        for q in filtered:
            if q.is_dir() and q.is_symlink():
                # 仍然显示这个链接，但不深入
                out.append(q)
            else:
                out.append(q)
        filtered = out

    return sorted(filtered, key=_sort_key)

def print_tree(
    root: str | Path,
    *,
    show_files: bool = True,
    max_depth: Optional[int] = None,
    ignore_globs: Optional[List[str]] = None,
    follow_symlinks: bool = False,
    prefix: str = "",
) -> None:
    """
    打印文件树结构。

    参数
    ----
    root : 根目录路径
    show_files : 是否显示文件（为 False 则只显示目录）
    max_depth : 最大深度（None 表示不限制；0 表示只显示 root 本身）
    ignore_globs : 忽略的通配符列表，例如 ["*.pyc", ".git", "__pycache__"]
    follow_symlinks : 是否深入符号链接目录
    prefix : 递归时内部使用的绘制前缀
    """
    root = Path(root)
    if max_depth is not None and max_depth < 0:
        return

    # 根行
    if prefix == "":
        print(root.resolve().as_posix())

    children = _children(root, show_files=show_files, follow_symlinks=follow_symlinks, ignore_globs=ignore_globs)
    # 仅保留目录（当 show_files=False）或全部
    if not show_files:
        children = [c for c in children if c.is_dir()]

    # 分支绘制
    for i, child in enumerate(children):
        connector = "└── " if i == len(children) - 1 else "├── "
        print(prefix + connector + child.name)

        # 递归
        if child.is_dir() and (follow_symlinks or not child.is_symlink()):
            if max_depth is None or max_depth > 0:
                extension = "    " if i == len(children) - 1 else "│   "
                print_tree(
                    child,
                    show_files=show_files,
                    max_depth=(None if max_depth is None else max_depth - 1),
                    ignore_globs=ignore_globs,
                    follow_symlinks=follow_symlinks,
                    prefix=prefix + extension,
                )

print_tree("/blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/2D_NS_FNO2d_recurrent/external_forcing_patterns/saved_models/", 
           show_files=True, max_depth=None, 
           ignore_globs=["*.gif","*.txt","*.png","*.ipynb","*.pkl","*.err","*.out","*.csv","*.sh","*.py"])


/blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/2D_NS_FNO2d_recurrent/external_forcing_patterns/saved_models
├── isoCircles
│   └── modes64_width60_epochs500_Tin10_T10
│       └── NS_2d_FNO_trained_on_dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0_train_forcingPatternisoCircles.pth
├── petals
│   └── modes64_width60_epochs500_Tin10_T10
│       └── NS_2d_FNO_trained_on_dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0_train_forcingPatternpetals.pth
├── ringsCos
│   └── modes64_width60_epochs500_Tin10_T10
│       └── NS_2d_FNO_trained_on_dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0_train_forcingPatternringsCos.pth
├── ringsL1
│   └── modes64_width60_epochs500_Tin10_T10
│       └── NS_2d_FNO_trained_on_dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0_train_forcingPatternringsL1.pth
├── ringsLinf
│   └── modes64_width60_epochs500_Tin10_T10
│       └── NS_2d_FNO_trained_on_dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0_train_forcingPatternringsLinf.pth
└── sBands
    └── modes64_wi

In [2]:
print_tree("/blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/2D_NS_FNO2d_recurrent/external_forcing_patterns/datasets/all_patterns", 
           show_files=True, max_depth=None, 
           ignore_globs=["*.gif","*.txt","*.png","*.ipynb","*.pkl","*.err","*.out","*.csv","*.sh","*.py"])


/blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/2D_NS_FNO2d_recurrent/external_forcing_patterns/datasets/all_patterns
├── isoCircles
│   ├── dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0_train_forcingPatternisoCircles.pt
│   └── dim2d_nx256_N50_solver=exponax_nu0.000_t20.0_test_forcingPatternisoCircles.pt
├── petals
│   ├── dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0_train_forcingPatternpetals.pt
│   └── dim2d_nx256_N50_solver=exponax_nu0.000_t20.0_test_forcingPatternpetals.pt
├── ringsCos
│   ├── dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0_train_forcingPatternringsCos.pt
│   └── dim2d_nx256_N50_solver=exponax_nu0.000_t20.0_test_forcingPatternringsCos.pt
├── ringsL1
│   ├── dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0_train_forcingPatternringsL1.pt
│   └── dim2d_nx256_N50_solver=exponax_nu0.000_t20.0_test_forcingPatternringsL1.pt
├── ringsLinf
│   ├── dim2d_nx256_N1150_solver=exponax_nu0.000_t20.0_train_forcingPatternringsLinf.pt
│   └── dim2d_nx256_N50_solver=expo

In [ ]:
print_tree("/blue/shiboli.fsu/yifeisun.umich/adversarial_robustness_FNO/2D_NS_FNO2d_recurrent/saved_models_kernel_abstracts/other_forcing_patterns/kernel_results", 
           show_files=True, max_depth=None, 
           ignore_globs=["*.gif","*.txt","*.png","*.ipynb","*.pkl","*.err","*.out","*.csv","*.sh","*.py"])
